## Introduction

In this lab assessment, you'll practice your knowledge of JOIN statements and subqueries, using various types of joins and various methods for specifying the links between them. One of the main benefits of using a relational database is the table relations that define them which allow you to access and connect data together via shared columns. By writing more advanced SQL queries that utilize joins and subqueries you can provide a deeper and more granular level of analysis and data retrieval.

This assessment will continue looking at the familiar Northwind database that contains customer relationship management (CRM) data as well as employee and product data. You will take a deeper dive into this database in order to accomplish more advanced SQL queries that require you to access data from multiple tables at once. 

Imagine that you are working in an analyst role for the sales rep team. They have collaborated with the customer relations and the product teams to take a comprehensive look at the employee to customer pipeline in an attempt to find areas of improvement and potential growth. You have been asked to provide some specific data and statistics regarding this project.

## Learning Objectives

You will be able to:

* Write SQL queries that make use of various types of joins
* Choose and perform whichever type of join is best for retrieving desired data
* Write subqueries to decompose complex queries

## Database

The database will be the customer relationship management (CRM) database, which has the following ERD.

![Database-Schema.png](ERD.png)

### Connect to the database

In the cell below we have provided the code to import both pandas and sqlite3 as well as define and create the connection to the database you will use. Also displayed is the schema and table names from the database. Use this information in conjunction with the ERD image above to assist in creating your SQL Queries.

Major Hint: Look for the shared columns across tables you need to 'join' together.

In [13]:
# CodeGrade step0
# Run this cell without changes

# SQL Library and Pandas Library
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect('data.sqlite')

pd.read_sql("""SELECT * FROM sqlite_master""", conn)

,type,name,tbl_name,rootpage,sql
0,table,orderdetails,orderdetails,2,"CREATE TABLE `orderdetails` (`orderNumber`, `p..."
1,table,payments,payments,28,"CREATE TABLE `payments` (`customerNumber`, `ch..."
2,table,offices,offices,32,"CREATE TABLE `offices` (`officeCode`, `city`, ..."
3,table,customers,customers,33,"CREATE TABLE `customers` (`customerNumber`, `c..."
4,table,orders,orders,38,"CREATE TABLE `orders` (`orderNumber`, `orderDa..."
5,table,productlines,productlines,46,"CREATE TABLE `productlines` (`productLine`, `t..."
6,table,products,products,47,"CREATE TABLE `products` (`productCode`, `produ..."
7,table,employees,employees,56,"CREATE TABLE `employees` (`employeeNumber`, `l..."


## Part 1: Join and Filter

### Step 1

The company would like to let Boston employees go remote but need to know more information about who is working in that office. Return the first and last names and the job titles for all employees in Boston.

In [14]:
# CodeGrade step1
# Query employees in Boston by joining employees and offices tables
# Filter results where the office city is 'Boston'
df_boston = pd.read_sql("""
    SELECT e.firstName, e.lastName
    FROM employees e
    JOIN offices o ON e.officeCode = o.officeCode
    WHERE o.city = 'Boston'
""", conn)

df_boston

,firstName,lastName,jobTitle
0,Julie,Firrelli,Sales Rep
1,Steve,Patterson,Sales Rep


### Step 2

Recent downsizing and employee attrition have caused some mixups in office tracking and the company is worried they are supporting a 'ghost' location. Are there any offices that have zero employees?

In [15]:
# CodeGrade step2
# Find offices with zero employees using a LEFT JOIN
# Offices with no matching employees will have NULL in the employee fields
df_zero_emp = pd.read_sql("""
    SELECT o.officeCode, o.city
    FROM offices o
    LEFT JOIN employees e ON o.officeCode = e.officeCode
    WHERE e.employeeNumber IS NULL
""", conn)

df_zero_emp

,officeCode,city


## Part 2: Type of Join

### Step 3

As a part of this larger analysis project the HR department is taking the time to audit employee records to make sure nothing is out of place and have asked you to produce a report of all employees. Return the employees first name and last name along with the city and state of the office that they work out of (if they have one). Include all employees and order them by their first name, then their last name.

In [16]:
# CodeGrade step3
# Return ALL employees with their office city and state using a LEFT JOIN
# LEFT JOIN ensures employees without an office are still included
# Order by first name, then last name alphabetically
df_employee = pd.read_sql("""
    SELECT e.firstName, e.lastName, o.city, o.state
    FROM employees e
    LEFT JOIN offices o ON e.officeCode = o.officeCode
    ORDER BY e.firstName, e.lastName
""", conn)

df_employee

,firstName,lastName,city,state
0,Andy,Fixter,Sydney,
1,Anthony,Bow,San Francisco,CA
2,Barry,Jones,London,
3,Diane,Murphy,San Francisco,CA
4,Foon Yue,Tseng,NYC,NY
5,George,Vanauf,NYC,NY
6,Gerard,Bondur,Paris,
7,Gerard,Hernandez,Paris,
8,Jeff,Firrelli,San Francisco,CA
9,Julie,Firrelli,Boston,MA


### Step 4
The customer management and sales rep team know that they have several 'customers' in the system that have not placed any orders. They want to reach out to these customers with updated product catalogs to try and get them to place initial orders. Return all of the customer's contact information (first name, last name, and phone number) as well as their sales rep's employee number for any customer that has not placed an order. Sort the results alphabetically based on the contact's last name

There are several approaches you could take here, including a left join and filtering on null values or using a subquery to filter out customers who do have orders. In total there are 24 customers who have not placed an order.

In [17]:
# CodeGrade step4
# Return contact info for customers who have NOT placed any orders
# LEFT JOIN customers to orders, then filter where no order exists (NULL)
# Sort alphabetically by contact last name
df_contacts = pd.read_sql("""
    SELECT c.contactFirstName, c.contactLastName, c.phone, c.salesRepEmployeeNumber
    FROM customers c
    LEFT JOIN orders o ON c.customerNumber = o.customerNumber
    WHERE o.orderNumber IS NULL
    ORDER BY c.contactLastName
""", conn)

df_contacts

,contactFirstName,contactLastName,phone,salesRepEmployeeNumber
0,Raanan,"Altagar,G M",+ 972 9 959 8555,
1,Mel,Andersen,030-0074555,
2,Carmen,Anton,+34 913 728555,
3,Alejandra,Camino,(91) 745 6555,
4,Philip,Cramer,0555-09555,
5,Alexander,Feuer,0342-555176,
6,Keith,Franco,2035557845,1286
7,Peter,Franken,089-0877555,
8,Ed,Harrison,+41 26 425 50 01,
9,Karin,Josephs,0251-555259,


## Part 3: Built-in Function

### Step 5

The accounting team is auditing their figures and wants to make sure all customer payments are in alignment, they have asked you to produce a report of all the customer contacts (first and last names) along with details for each of the customers' payment amounts and date of payment. They have asked that these results be sorted in descending order by the payment amount.

Hint: A member of their team mentioned that they are not sure the 'amount' column is being stored as the right datatype so keep this in mind when sorting.

In [18]:
# CodeGrade step5
# Return customer contact names with their payment amounts and dates
# JOIN customers to payments using the shared customerNumber column
# CAST amount to REAL to ensure correct numeric sorting (not alphabetical)
# Sort by payment amount in descending order
df_payment = pd.read_sql("""
    SELECT c.contactFirstName, c.contactLastName, p.amount, p.paymentDate
    FROM customers c
    JOIN payments p ON c.customerNumber = p.customerNumber
    ORDER BY CAST(p.amount AS REAL) DESC
""", conn)

df_payment

,contactFirstName,contactLastName,amount,paymentDate
0,Diego,Freyre,120166.58,2005-03-18
1,Diego,Freyre,116208.40,2004-12-31
2,Susan,Nelson,111654.40,2003-08-15
3,Eric,Natividad,105743.00,2003-12-26
4,Susan,Nelson,101244.59,2005-03-05
...,...,...,...,...
268,Carine,Schmitt,1676.14,2004-12-18
269,Pascale,Cartrain,1627.56,2003-04-19
270,Jonas,Bergulfsen,1491.38,2003-10-28
271,Pascale,Cartrain,1128.20,2003-08-22


## Part 4: Joining and Grouping

### Step 6

The sales rep team has noticed several key team members that stand out as having trustworthy business relations with their customers, reflected by high credit limits indicating more potential for orders. The team wants you to identify these 4 individuals. Return the employee number, first name, last name, and number of customers for employees whose customers have an average credit limit over 90k. Sort by number of customers from high to low.

In [19]:
# CodeGrade step6
# Return employees whose customers have an average credit limit over $90,000
# JOIN employees to customers using salesRepEmployeeNumber as the foreign key
# GROUP BY employee, use HAVING to filter groups where avg credit limit exceeds 90k
# Sort by number of customers from highest to lowest
df_credit = pd.read_sql("""
    SELECT e.employeeNumber, e.firstName, e.lastName, COUNT(c.customerNumber) AS numCustomers
    FROM employees e
    JOIN customers c ON e.employeeNumber = c.salesRepEmployeeNumber
    GROUP BY e.employeeNumber, e.firstName, e.lastName
    HAVING AVG(CAST(c.creditLimit AS REAL)) > 90000
    ORDER BY numCustomers DESC
""", conn)

df_credit

,employeeNumber,firstName,lastName,numCustomers
0,1501,Larry,Bott,8
1,1370,Gerard,Hernandez,7
2,1165,Leslie,Jennings,6
3,1612,Peter,Marsh,5


### Step 7

The product team is looking to create new model kits and wants to know which current products are selling the most in order to get an idea of what is popular. Return the product name and count the number of orders for each product as a column named 'numorders'. Also return a new column, 'totalunits', that sums up the total quantity of product sold (use the quantityOrdered column). Sort the results by the totalunits column, highest to lowest, to showcase the top selling products.

In [20]:
# CodeGrade step7
# Return each product's name, number of orders, and total quantity sold
# JOIN products to orderdetails using productCode
# COUNT orders as 'numorders', SUM quantityOrdered as 'totalunits'
# GROUP BY product and sort by totalunits highest to lowest
df_product_sold = pd.read_sql("""
    SELECT p.productName,
           COUNT(od.orderNumber) AS numorders,
           SUM(od.quantityOrdered) AS totalunits
    FROM products p
    JOIN orderdetails od ON p.productCode = od.productCode
    GROUP BY p.productCode, p.productName
    ORDER BY totalunits DESC
""", conn)

df_product_sold

,productName,numorders,totalunits
0,1992 Ferrari 360 Spider red,53,1808
1,1937 Lincoln Berline,28,1111
2,American Airlines: MD-11S,28,1085
3,1941 Chevrolet Special Deluxe Cabriolet,28,1076
4,1930 Buick Marquette Phaeton,28,1074
...,...,...,...
104,1999 Indy 500 Monte Carlo SS,25,855
105,1911 Ford Town Car,25,832
106,1936 Mercedes Benz 500k Roadster,25,824
107,1970 Chevy Chevelle SS 454,25,803


## Part 5: Multiple Joins

### Step 8

As a follow-up to the above question, the product team also wants to know how many different customers ordered each product to get an idea of market reach. Return the product name, code, and the total number of customers who have ordered each product, aliased as 'numpurchasers'. Sort the results by the highest  number of purchasers.

Hint: You might need to join more than 2 tables. Use DISTINCT to return unique/different values.

In [21]:
# CodeGrade step8
# Return each product's name, code, and number of distinct customers who ordered it
# Chain joins: products -> orderdetails -> orders -> customers
# Use COUNT(DISTINCT) to avoid counting the same customer multiple times per product
# Sort by highest number of purchasers
df_total_customers = pd.read_sql("""
    SELECT p.productName, p.productCode,
           COUNT(DISTINCT c.customerNumber) AS numpurchasers
    FROM products p
    JOIN orderdetails od ON p.productCode = od.productCode
    JOIN orders o ON od.orderNumber = o.orderNumber
    JOIN customers c ON o.customerNumber = c.customerNumber
    GROUP BY p.productCode, p.productName
    ORDER BY numpurchasers DESC
""", conn)

df_total_customers

,productName,productCode,numpurchasers
0,1992 Ferrari 360 Spider red,S18_3232,40
1,1952 Alpine Renault 1300,S10_1949,27
2,1972 Alfa Romeo GTA,S10_4757,27
3,1934 Ford V8 Coupe,S18_2957,27
4,Boeing X-32A JSF,S72_1253,27
...,...,...,...
104,1958 Chevy Corvette Limited Edition,S24_2840,19
105,1949 Jaguar XK 120,S24_2766,18
106,1952 Citroen-15CV,S24_2887,18
107,1969 Chevrolet Camaro Z28,S24_3191,18


### Step 9

The custom relations team is worried they are not staffing locations properly to account for customer volume. They want to know how many customers there are per office. Return the count as a column named 'n_customers'. Also return the office code and city.

In [22]:
# CodeGrade step9
# Return the number of customers per office to assess staffing levels
# Chain joins: offices -> employees -> customers using officeCode and salesRepEmployeeNumber
# COUNT distinct customers per office, return office code and city
# Sort by highest number of customers
df_customers = pd.read_sql("""
    SELECT o.officeCode, o.city,
           COUNT(DISTINCT c.customerNumber) AS n_customers
    FROM offices o
    JOIN employees e ON o.officeCode = e.officeCode
    JOIN customers c ON e.employeeNumber = c.salesRepEmployeeNumber
    GROUP BY o.officeCode, o.city
    ORDER BY n_customers DESC
""", conn)

df_customers

,officeCode,city,n_customers
0,4,Paris,29
1,7,London,17
2,3,NYC,15
3,1,San Francisco,12
4,2,Boston,12
5,6,Sydney,10
6,5,Tokyo,5


## Part 6: Subquery

### Step 10

Having looked at the results from above, the product team is curious to dig into the underperforming products. They want to ask members of the team who have sold these products about what kind of messaging was successful in getting a customer to buy these specific products. Using a subquery or common table expression (CTE), select the employee number, first name, last name, city of the office, and the office code for employees who sold products that have been ordered by fewer than 20 customers.

Hint: Start with the subquery, find all the products that have been ordered by 19 or less customers, consider adapting one of your previous queries.

In [23]:
# CodeGrade step10
# Return employees who sold products ordered by fewer than 20 distinct customers
# Subquery: find all productCodes with fewer than 20 distinct purchasers (adapted from step 8)
# Outer query: trace the sales pipeline to find which employees sold those products
# Chain joins: employees -> orders -> orderdetails, filtered by the subquery
# Use DISTINCT to avoid duplicate employee rows, also return their office city and code
df_under_20 = pd.read_sql("""
    SELECT DISTINCT e.employeeNumber, e.firstName, e.lastName, o.city, o.officeCode
    FROM employees e
    JOIN offices o ON e.officeCode = o.officeCode
    JOIN customers c ON e.employeeNumber = c.salesRepEmployeeNumber
    JOIN orders ord ON c.customerNumber = ord.customerNumber
    JOIN orderdetails od ON ord.orderNumber = od.orderNumber
    WHERE od.productCode IN (
        -- Subquery: products ordered by fewer than 20 distinct customers
        SELECT p.productCode
        FROM products p
        JOIN orderdetails od2 ON p.productCode = od2.productCode
        JOIN orders o2 ON od2.orderNumber = o2.orderNumber
        JOIN customers c2 ON o2.customerNumber = c2.customerNumber
        GROUP BY p.productCode
        HAVING COUNT(DISTINCT c2.customerNumber) < 20
    )
    ORDER BY e.firstName DESC
""", conn)

df_under_20

,employeeNumber,firstName,lastName,city,officeCode
0,1370,Gerard,Hernandez,Paris,4
1,1501,Larry,Bott,London,7
2,1337,Loui,Bondur,Paris,4
3,1166,Leslie,Thompson,San Francisco,1
4,1286,Foon Yue,Tseng,NYC,3
5,1612,Peter,Marsh,Sydney,6
6,1611,Andy,Fixter,Sydney,6
7,1401,Pamela,Castillo,Paris,4
8,1621,Mami,Nishi,Tokyo,5
9,1323,George,Vanauf,NYC,3


### Close the connection

In [ ]:
# Run this cell without changes

conn.close()